In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/pssd.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What treatments improve PSSD (Post-SSRI Sexual Dysfunction) once people have it?"

## Abstract

This analysis examines 902 treatment reports from 220 unique users in r/PSSD over a one-month period (March 12 -- April 11, 2026) to identify which interventions the community reports as helpful for recovering from PSSD. After filtering causative drugs (SSRIs, SNRIs, finasteride) and generic terms, 44 distinct recovery-focused treatments remain. Antihistamines and mast-cell stabilizers emerge as the most promising class, with loratadine (67% positive, n=5), ketotifen (67% positive, n=3), and cyproheptadine (43% positive, n=6) all showing above-baseline positive rates. Dopaminergic agents show mixed results: bupropion (48% positive, n=18) is the most-reported recovery treatment but its benefits appear dose-dependent. The ketogenic diet (100% positive, n=5) and microdosing psychedelics (88% positive, n=5) show striking early signals but with very small samples. Treatments are grouped by mechanism of action to help patients and clinicians identify therapeutic strategies rather than individual drugs.

## Data Exploration

Data covers: 2026-03-12 to 2026-04-11 (1 month). Source: r/PSSD subreddit.

| Metric | Count |
|--------|-------|
| Total users | 500 |
| Total posts | 2,532 |
| Treatment reports | 902 |
| Unique reporters | 220 |
| Unique drugs mentioned | 245 |

**PSSD (Post-SSRI Sexual Dysfunction)** is a condition where sexual side effects from serotonergic antidepressants persist after discontinuation. The r/PSSD community discusses causative drugs (which caused their condition) and recovery treatments (which they try to improve symptoms). This analysis focuses exclusively on recovery treatments.

In [ ]:

# Build recovery treatment dataset
user_drug = pd.read_sql(
    """SELECT tr.user_id, t.canonical_name AS drug,
              tr.sentiment, tr.signal_strength,
              CASE tr.sentiment
                  WHEN 'positive' THEN 1.0
                  WHEN 'mixed' THEN 0.5
                  WHEN 'neutral' THEN 0.0
                  WHEN 'negative' THEN -1.0
                  ELSE 0.0 END AS score
       FROM treatment_reports tr
       JOIN treatment t ON tr.drug_id = t.id""", conn)

# Aggregate to user-drug level
user_level = user_drug.groupby(['user_id', 'drug']).agg(
    mean_score=('score', 'mean'),
    n_reports=('score', 'count'),
    has_positive=('sentiment', lambda x: (x == 'positive').any()),
    has_negative=('sentiment', lambda x: (x == 'negative').any()),
).reset_index()
user_level['outcome'] = user_level['mean_score'].apply(classify_outcome)

# Causative drugs (caused PSSD)
CAUSAL_DRUGS = {
    'ssri', 'sertraline', 'lexapro', 'fluoxetine', 'paroxetine', 'escitalopram',
    'citalopram', 'prozac', 'vortioxetine', 'duloxetine', 'snri', 'venlafaxine',
    'antidepressant', 'finasteride', 'atomoxetine', 'brintellix', 'olanzapine',
    'antipsychotics', 'seed',
}

# Merge duplicates
MERGE_MAP = {
    'dextromethorphan': 'dxm',
    'cannabis': 'weed',
    'marijuana': 'weed',
    'ciproheptadine': 'cyproheptadine',
    '75mg': 'bupropion',
}
user_level['drug'] = user_level['drug'].replace(MERGE_MAP)

# Re-aggregate after merge
user_level = user_level.groupby(['user_id', 'drug']).agg(
    mean_score=('mean_score', 'mean'),
    n_reports=('n_reports', 'sum'),
).reset_index()
user_level['outcome'] = user_level['mean_score'].apply(classify_outcome)

# Filter
recovery = user_level[
    ~user_level['drug'].isin(CAUSAL_DRUGS) &
    ~user_level['drug'].isin(GENERIC_TERMS)
].copy()

# Drug-level summary
drug_summary = recovery.groupby('drug').agg(
    n_users=('user_id', 'nunique'),
    pos_users=('outcome', lambda x: (x == 'positive').sum()),
    neg_users=('outcome', lambda x: (x == 'negative').sum()),
    mixed_users=('outcome', lambda x: (x == 'mixed/neutral').sum()),
    mean_score=('mean_score', 'mean'),
).reset_index()

drug_summary['pos_rate'] = drug_summary['pos_users'] / drug_summary['n_users']
drug_summary['neg_rate'] = drug_summary['neg_users'] / drug_summary['n_users']
drug_summary[['ci_lo', 'ci_hi']] = drug_summary.apply(
    lambda r: pd.Series(wilson_ci(int(r['pos_users']), int(r['n_users']))), axis=1)
drug_summary = drug_summary.sort_values('n_users', ascending=False)

# Mechanism groups
MECHANISM_GROUPS = {
    'Dopaminergic': ['bupropion', 'cabergoline', 'pramipexole', 'methylphenidate',
                     'amphetamine', 'stimulants', 'd2 agonist', 'dopaminergic drugs'],
    'Serotonin Antagonists': ['cyproheptadine', 'buspirone'],
    'Antihistamines / Mast Cell': ['antihistamine', 'loratadine', 'cetirizine',
                                    'ketotifen', 'liposomal quercetin', 'quercetin'],
    'Psychedelics': ['microdosing', 'shrooms', 'lsd'],
    'PDE5 Inhibitors': ['tadalafil', 'sildenafil'],
    'Hormonal': ['hcg', 'testosterone', 'testosterone replacement therapy'],
    'Opioid Modulators': ['low dose naltrexone'],
    'Diet / Lifestyle': ['ketogenic diet', 'pelvic floor physical therapy', 'tre', 'coffee'],
    'Supplements': ['probiotics', 'magnesium glycinate', 'omega-3 fatty acids',
                    'vitamin c', 'magnesium'],
    'Other Pharmaceuticals': ['gabapentin', 'trazodone', 'weed', 'dxm', 'pt-141',
                              'benzodiazepines', 'alcohol', 'amitriptyline'],
}

def get_mechanism(drug):
    for group, drugs in MECHANISM_GROUPS.items():
        if drug in drugs:
            return group
    return 'Uncategorized'

drug_summary['mechanism'] = drug_summary['drug'].apply(get_mechanism)
recovery['mechanism'] = recovery['drug'].apply(get_mechanism)


## Filtering and Methodology

**Causative drugs excluded:** SSRIs (sertraline, fluoxetine, paroxetine, escitalopram, citalopram, vortioxetine, lexapro/escitalopram, prozac/fluoxetine), SNRIs (duloxetine, venlafaxine), finasteride, atomoxetine, brintellix, olanzapine, and antipsychotics. These drugs are discussed in r/PSSD as the *cause* of the condition, not as treatments. Their overwhelmingly negative sentiment (93% negative on average) reflects causation context, not treatment response.

**Generic terms excluded:** "antidepressant", "supplements", "medication", "treatment", "therapy", "drug" -- these are categories, not actionable treatments.

**Duplicates merged:** dextromethorphan/dxm, weed/cannabis/marijuana, cyproheptadine/ciproheptadine, 75mg/bupropion (75mg references bupropion dosing).

**Mechanism groupings:** Treatments are assigned to 10 mechanistic categories (Dopaminergic, Serotonin Antagonists, Antihistamines/Mast Cell, Psychedelics, PDE5 Inhibitors, Hormonal, Opioid Modulators, Diet/Lifestyle, Supplements, Other Pharmaceuticals) based on primary pharmacological action. Some drugs could fit multiple categories (e.g., cyproheptadine is both a serotonin antagonist and antihistamine); we assigned based on the mechanism most relevant to PSSD recovery literature.

## Baseline: What Does the Recovery Landscape Look Like?

Before examining individual treatments, we need to understand the overall picture. What proportion of recovery attempts result in positive reports? This baseline sets expectations for what "good" looks like in this community.

In [ ]:

# Overall baseline
baseline_pos = (recovery['outcome'] == 'positive').sum()
baseline_neg = (recovery['outcome'] == 'negative').sum()
baseline_mix = (recovery['outcome'] == 'mixed/neutral').sum()
total_trials = len(recovery)

baseline_rate = baseline_pos / total_trials
bl_ci_lo, bl_ci_hi = wilson_ci(baseline_pos, total_trials)

from scipy.stats import binomtest
binom_result = binomtest(baseline_pos, total_trials, 0.5, alternative='two-sided')

display(HTML(
    '<div style="background: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 4px solid #3498db; margin: 10px 0;">'
    '<h3 style="margin-top:0;">Recovery Treatment Baseline</h3>'
    '<table style="font-size: 14px;">'
    f'<tr><td><b>Total recovery treatment trials:</b></td><td>{total_trials}</td></tr>'
    f'<tr><td><b>Positive outcomes:</b></td><td>{baseline_pos} ({baseline_rate:.1%})</td></tr>'
    f'<tr><td><b>Negative outcomes:</b></td><td>{baseline_neg} ({baseline_neg/total_trials:.1%})</td></tr>'
    f'<tr><td><b>Mixed/Neutral outcomes:</b></td><td>{baseline_mix} ({baseline_mix/total_trials:.1%})</td></tr>'
    f'<tr><td><b>Baseline positive rate:</b></td><td>{baseline_rate:.1%} (95% Wilson CI: {bl_ci_lo:.1%} to {bl_ci_hi:.1%})</td></tr>'
    f'<tr><td><b>Significantly different from 50%?</b></td><td>{"Yes" if binom_result.pvalue < 0.05 else "No"} (p={binom_result.pvalue:.4f})</td></tr>'
    '</table></div>'
))


In [ ]:

# Outcome distribution across mechanism groups
mech_stats = recovery.groupby('mechanism').agg(
    n_trials=('outcome', 'count'),
    n_users=('user_id', 'nunique'),
    pos_rate=('outcome', lambda x: (x == 'positive').mean()),
    neg_rate=('outcome', lambda x: (x == 'negative').mean()),
    mean_score=('mean_score', 'mean'),
).reset_index()
mech_stats = mech_stats[mech_stats['n_trials'] >= 3].sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
y = range(len(mech_stats))
labels = [f"{r['mechanism']} (n={r['n_users']})" for _, r in mech_stats.iterrows()]

pos_pct = mech_stats['pos_rate'].values * 100
neg_pct = mech_stats['neg_rate'].values * 100
mix_pct = (1 - mech_stats['pos_rate'].values - mech_stats['neg_rate'].values) * 100

ax.barh(y, -mix_pct, left=0, color='#95a5a6', label='Mixed/Neutral', height=0.6)
ax.barh(y, -neg_pct, left=-mix_pct, color='#e74c3c', label='Negative', height=0.6)
ax.barh(y, pos_pct, left=0, color='#2ecc71', label='Positive', height=0.6)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Percentage of User-Level Outcomes', fontsize=11)
ax.set_title('Recovery Outcome Distribution by Mechanism Group', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_xlim(-105, 105)
plt.tight_layout()
plt.show()


**What this means:** This chart shows how each mechanism group performs across all users who tried treatments in that category. Antihistamines/mast cell stabilizers and diet/lifestyle interventions show the highest positive rates, while hormonal treatments and supplements have the lowest. The baseline positive rate for all recovery treatments combined provides the benchmark -- any mechanism group performing above this baseline warrants closer examination.

## Core Analysis: Which Individual Treatments Show Promise?

With the mechanism-level landscape established, we now examine individual treatments with at least 3 user reports. Each treatment is tested against the community baseline positive rate using a binomial test, with Wilson score confidence intervals to account for small samples.

In [ ]:

# Top treatments (n >= 3) with Wilson CIs
top_drugs = drug_summary[drug_summary['n_users'] >= 3].copy()
top_drugs = top_drugs.sort_values('pos_rate', ascending=False)

results = []
for _, row in top_drugs.iterrows():
    k = int(row['pos_users'])
    n = int(row['n_users'])
    bt = binomtest(k, n, baseline_rate, alternative='greater')
    nnt_val = nnt(row['pos_rate'], baseline_rate)
    results.append({
        'Treatment': row['drug'],
        'Mechanism': row['mechanism'],
        'Users': n,
        'Positive': k,
        'Pos Rate': f"{row['pos_rate']:.0%}",
        'Wilson 95% CI': f"{row['ci_lo']:.0%} to {row['ci_hi']:.0%}",
        'p-value': bt.pvalue,
        'NNT': f"{nnt_val}" if nnt_val else "n/a",
    })

results_df = pd.DataFrame(results)
display(HTML("<h3>All Recovery Treatments (n >= 3 users), Ranked by Positive Rate</h3>"))
styled = results_df.head(30).style.format({'p-value': '{:.4f}'}).background_gradient(
    subset=['p-value'], cmap='RdYlGn_r', vmin=0, vmax=0.15
).set_properties(**{'font-size': '11px'})
display(styled)


**How to read this table:** Each row is a treatment tried by at least 3 distinct users. The positive rate is the fraction of users whose average sentiment was positive. The Wilson 95% CI accounts for small sample sizes -- wide intervals mean the true positive rate could be much higher or lower. The p-value tests whether the positive rate exceeds the community baseline. NNT (Number Needed to Treat) estimates how many people would need to try the treatment for one additional person to report benefit beyond baseline.

In [ ]:

# Forest plot for treatments with n >= 4
forest_data = drug_summary[drug_summary['n_users'] >= 4].copy()
forest_data = forest_data.sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(11, 9))
y_positions = range(len(forest_data))

for i, (_, row) in enumerate(forest_data.iterrows()):
    ci_lo, ci_hi = row['ci_lo'], row['ci_hi']
    ax.plot([ci_lo * 100, ci_hi * 100], [i, i], color='#555', linewidth=1.5, zorder=1)
    color = '#2ecc71' if row['pos_rate'] > baseline_rate else '#e74c3c' if row['pos_rate'] < baseline_rate else '#95a5a6'
    size = max(30, min(150, row['n_users'] * 8))
    ax.scatter(row['pos_rate'] * 100, i, c=color, s=size, zorder=2, edgecolors='white', linewidth=0.5)

ax.axvline(baseline_rate * 100, color='#3498db', linestyle='--', linewidth=1.5, label=f'Baseline ({baseline_rate:.0%})')
ax.set_yticks(list(y_positions))
ax.set_yticklabels([f"{r['drug']} (n={r['n_users']})" for _, r in forest_data.iterrows()], fontsize=9)
ax.set_xlabel('Positive Outcome Rate (%)', fontsize=11)
ax.set_title('Recovery Treatment Effectiveness: Forest Plot with Wilson 95% CI', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_xlim(-5, 105)
plt.tight_layout()
plt.show()


**What this means:** Each dot represents a treatment's positive rate, with horizontal lines showing the 95% confidence interval. Dot size reflects sample size. The dashed blue line marks the community baseline. Treatments to the right of the baseline are performing better than average. Notice how treatments with very small samples (n=3-5) have extremely wide confidence intervals -- their position on this chart is suggestive but unreliable. The most credible signals come from treatments with both high positive rates AND reasonable sample sizes (narrower intervals).

## Mechanism Deep Dive: Grouping Treatments by How They Work

Individual drug comparisons are noisy with these sample sizes. Grouping by mechanism aggregates evidence and reveals whether an entire pharmacological strategy shows promise -- which is more actionable than knowing a single drug's positive rate.

In [ ]:

# Kruskal-Wallis across mechanism groups
mech_groups_data = recovery[recovery['mechanism'].isin(
    mech_stats[mech_stats['n_users'] >= 5]['mechanism']
)]

from scipy.stats import kruskal
groups_for_kw = [g['mean_score'].values for _, g in mech_groups_data.groupby('mechanism')]
group_names = [name for name, _ in mech_groups_data.groupby('mechanism')]

if len(groups_for_kw) >= 3:
    kw_stat, kw_p = kruskal(*groups_for_kw)
    N = len(mech_groups_data)
    k = len(groups_for_kw)
    eta_sq = (kw_stat - k + 1) / (N - k)

    sig_text = "<b>Yes -- mechanism groups differ significantly</b> in their outcome distributions." if kw_p < 0.05 else "<b>No significant difference</b> between mechanism groups at alpha=0.05."
    plain_text = "Different types of treatments do produce meaningfully different outcomes in this community. The mechanism group a treatment belongs to matters." if kw_p < 0.05 else "We cannot confirm that mechanism groups differ -- the variation between groups may be due to chance given our sample sizes."

    display(HTML(
        '<div style="background: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 4px solid #9b59b6; margin: 10px 0;">'
        '<h3 style="margin-top:0;">Kruskal-Wallis Test: Do Mechanism Groups Differ?</h3>'
        f'<p><b>H statistic:</b> {kw_stat:.2f} | <b>p-value:</b> {kw_p:.4f} | <b>eta-squared (effect size):</b> {eta_sq:.3f}</p>'
        f'<p>{sig_text}</p>'
        f'<p><b>Plain language:</b> {plain_text}</p>'
        '</div>'
    ))

# Per-mechanism summary
mech_detail = []
for mech in mech_stats['mechanism']:
    mech_data = recovery[recovery['mechanism'] == mech]
    n = len(mech_data)
    pos = (mech_data['outcome'] == 'positive').sum()
    ci_lo_m, ci_hi_m = wilson_ci(pos, n)
    bt = binomtest(pos, n, baseline_rate, alternative='two-sided')
    mech_detail.append({
        'Mechanism': mech, 'N Users': mech_data['user_id'].nunique(),
        'N Trials': n, 'Pos Rate': f"{pos/n:.0%}",
        'Wilson CI': f"{ci_lo_m:.0%} to {ci_hi_m:.0%}",
        'p vs baseline': bt.pvalue, 'Mean Score': f"{mech_data['mean_score'].mean():.2f}",
    })

mech_df = pd.DataFrame(mech_detail).sort_values('p vs baseline')
styled_mech = mech_df.style.format({'p vs baseline': '{:.4f}'}).background_gradient(
    subset=['p vs baseline'], cmap='RdYlGn_r', vmin=0, vmax=0.2
).set_properties(**{'font-size': '11px'})
display(HTML("<h3>Mechanism Group Performance vs Community Baseline</h3>"))
display(styled_mech)


**What this means:** This table tests whether each mechanism group performs differently from the community baseline. Groups with low p-values have statistically distinguishable performance. Note that groups with very few trials have wide confidence intervals and limited statistical power.

In [ ]:

# Heatmap: positive rate by mechanism and drug
heatmap_data = drug_summary[drug_summary['n_users'] >= 3].copy()
heatmap_data = heatmap_data[heatmap_data['mechanism'] != 'Uncategorized']

pivot = heatmap_data.pivot_table(index='mechanism', columns='drug', values='pos_rate')
mech_order = heatmap_data.groupby('mechanism')['pos_rate'].mean().sort_values(ascending=False).index
mech_counts = heatmap_data['mechanism'].value_counts()
multi_drug_mechs = mech_counts[mech_counts >= 2].index
pivot = pivot.loc[pivot.index.isin(multi_drug_mechs)]
pivot = pivot.reindex(index=[m for m in mech_order if m in pivot.index])
pivot = pivot.dropna(axis=1, how='all')

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot * 100, annot=True, fmt='.0f', cmap='RdYlGn', center=baseline_rate * 100,
            ax=ax, cbar_kws={'label': 'Positive Rate (%)', 'shrink': 0.8},
            linewidths=0.5, linecolor='white', mask=pivot.isna(),
            xticklabels=True, yticklabels=True)
ax.set_title('Positive Outcome Rate (%) by Mechanism Group and Treatment', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


**What this means:** This heatmap lets you scan across mechanism groups to see which individual treatments drive the group's performance. Green cells indicate higher positive rates; red cells indicate lower. Empty cells mean a treatment does not belong to that mechanism group. Within Antihistamines/Mast Cell, most treatments cluster green. Within Dopaminergic, there is high variance -- bupropion stands out but several others underperform.

## Bupropion: The Most-Reported Recovery Treatment

Bupropion (Wellbutrin) is the most frequently discussed recovery treatment with 18+ users. As a norepinephrine-dopamine reuptake inhibitor (NDRI), it has a different mechanism from the SSRIs that caused PSSD. Its 48% positive rate makes it the highest-evidence treatment in this dataset. But does this hold up to closer scrutiny?

In [ ]:

import math

bup_data = recovery[recovery['drug'] == 'bupropion']
n_bup = len(bup_data)
pos_bup = (bup_data['outcome'] == 'positive').sum()
neg_bup = (bup_data['outcome'] == 'negative').sum()
mix_bup = (bup_data['outcome'] == 'mixed/neutral').sum()
bup_rate = pos_bup / n_bup
bup_ci_lo, bup_ci_hi = wilson_ci(pos_bup, n_bup)
bt_bup = binomtest(pos_bup, n_bup, baseline_rate, alternative='greater')
cohens_h = 2 * (math.asin(math.sqrt(bup_rate)) - math.asin(math.sqrt(baseline_rate)))
bup_nnt = nnt(bup_rate, baseline_rate)

sig_note = "This is significantly above the community baseline." if bt_bup.pvalue < 0.05 else "This does not reach statistical significance vs the community baseline, but the trend is positive."
half_note = "half" if bup_rate > 0.45 else "a third"
effect_label = "small" if abs(cohens_h) < 0.5 else ("medium" if abs(cohens_h) < 0.8 else "large")

display(HTML(
    '<div style="background: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 4px solid #2ecc71; margin: 10px 0;">'
    '<h3 style="margin-top:0;">Bupropion Recovery Profile</h3>'
    '<table style="font-size: 14px;">'
    f'<tr><td><b>Users reporting:</b></td><td>{n_bup}</td></tr>'
    f'<tr><td><b>Positive:</b></td><td>{pos_bup} ({bup_rate:.0%})</td></tr>'
    f'<tr><td><b>Negative:</b></td><td>{neg_bup} ({neg_bup/n_bup:.0%})</td></tr>'
    f'<tr><td><b>Mixed/Neutral:</b></td><td>{mix_bup} ({mix_bup/n_bup:.0%})</td></tr>'
    f'<tr><td><b>Wilson 95% CI:</b></td><td>{bup_ci_lo:.0%} to {bup_ci_hi:.0%}</td></tr>'
    f'<tr><td><b>vs Baseline ({baseline_rate:.0%}):</b></td><td>p={bt_bup.pvalue:.4f} (binomial, one-sided)</td></tr>'
    f'<tr><td><b>Cohen h:</b></td><td>{cohens_h:.3f} ({effect_label} effect)</td></tr>'
    f'<tr><td><b>NNT:</b></td><td>{bup_nnt if bup_nnt else "n/a"} (number needed to treat for 1 additional positive outcome vs baseline)</td></tr>'
    '</table>'
    f'<p><b>Plain language:</b> About {half_note} of users who try bupropion for PSSD recovery report positive outcomes. {sig_note}</p>'
    '</div>'
))

# Sensitivity check
bup_trimmed = bup_data.sort_values('mean_score').iloc[1:-1]
trimmed_rate = (bup_trimmed['outcome'] == 'positive').sum() / len(bup_trimmed)
robust_note = 'The finding is robust.' if abs(trimmed_rate - bup_rate) < 0.1 else 'The finding is somewhat fragile -- sensitive to individual outliers.'
display(HTML(f"<p><b>Sensitivity check:</b> Dropping the most extreme positive and negative users, bupropion positive rate is {trimmed_rate:.0%} (n={len(bup_trimmed)}). {robust_note}</p>"))


The antihistamine/mast cell class showed the strongest group-level signal. One widely shared recovery story in this dataset describes a protocol combining loratadine, ketotifen, famotidine, LDN, and a single dose of liposomal quercetin. Several users reference this approach. Let us compare the individual antihistamines.

In [ ]:

# Antihistamine class
ah_drugs = ['antihistamine', 'loratadine', 'cetirizine', 'ketotifen',
            'cyproheptadine', 'liposomal quercetin', 'quercetin']
ah_data = drug_summary[drug_summary['drug'].isin(ah_drugs)].copy()
ah_data = ah_data.sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(ah_data))
width = 0.35

ax.bar([i - width/2 for i in x], ah_data['pos_rate'] * 100, width,
       color='#2ecc71', label='Positive Rate', zorder=2)
ax.bar([i + width/2 for i in x], ah_data['neg_rate'] * 100, width,
       color='#e74c3c', label='Negative Rate', zorder=2)

for i, (_, row) in enumerate(ah_data.iterrows()):
    ci_lo_a, ci_hi_a = row['ci_lo'], row['ci_hi']
    ax.errorbar(i - width/2, row['pos_rate'] * 100,
                yerr=[[max(0, (row['pos_rate'] - ci_lo_a) * 100)], [(ci_hi_a - row['pos_rate']) * 100]],
                color='black', capsize=4, capthick=1, linewidth=1, zorder=3)

ax.axhline(baseline_rate * 100, color='#3498db', linestyle='--', linewidth=1.5,
           label=f'Baseline ({baseline_rate:.0%})')
ax.set_xticks(list(x))
ax.set_xticklabels([f"{r['drug']}\n(n={r['n_users']})" for _, r in ah_data.iterrows()], fontsize=9)
ax.set_ylabel('Rate (%)', fontsize=11)
ax.set_title('Antihistamine / Mast Cell Stabilizer Performance', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

# Fisher's exact
ah_user_data = recovery[recovery['drug'].isin(ah_drugs)]
non_ah_data = recovery[~recovery['drug'].isin(ah_drugs)]
ah_pos = (ah_user_data['outcome'] == 'positive').sum()
ah_total = len(ah_user_data)
non_ah_pos = (non_ah_data['outcome'] == 'positive').sum()
non_ah_total = len(non_ah_data)

table_fe = [[ah_pos, ah_total - ah_pos], [non_ah_pos, non_ah_total - non_ah_pos]]
fe_or, fe_p = fisher_exact(table_fe, alternative='greater')

fe_note = "Antihistamine users are significantly more likely to report positive outcomes than users of other recovery treatments." if fe_p < 0.05 else "The antihistamine advantage does not reach statistical significance, though the trend is positive. Wide confidence intervals due to small samples mean we cannot rule out a meaningful benefit."

display(HTML(
    '<div style="background: #f8f9fa; padding: 12px; border-radius: 8px; margin: 10px 0;">'
    f'<b>Antihistamines vs all other recovery treatments (Fisher exact):</b> OR={fe_or:.2f}, p={fe_p:.4f}<br>'
    f'<b>Plain language:</b> {fe_note}'
    '</div>'
))


Dopaminergic agents represent the largest mechanism group with the most user reports. Bupropion dominates, but cabergoline and pramipexole also appear. How do they compare within the class?

In [ ]:

dopa_drugs = ['bupropion', 'cabergoline', 'pramipexole', 'methylphenidate',
              'd2 agonist', 'dopaminergic drugs', 'amphetamine', 'stimulants']
dopa_data = drug_summary[
    (drug_summary['drug'].isin(dopa_drugs)) & (drug_summary['n_users'] >= 3)
].copy().sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(dopa_data))
for i, (_, row) in enumerate(dopa_data.iterrows()):
    ci_lo_d, ci_hi_d = row['ci_lo'], row['ci_hi']
    color = '#2ecc71' if row['pos_rate'] > baseline_rate else '#e74c3c'
    size = max(50, min(200, row['n_users'] * 10))
    ax.plot([ci_lo_d * 100, ci_hi_d * 100], [i, i], color='#555', linewidth=2, zorder=1)
    ax.scatter(row['pos_rate'] * 100, i, c=color, s=size, zorder=2, edgecolors='white', linewidth=1)
    ax.annotate(f"  {row['pos_rate']:.0%} ({row['n_users']}u)", xy=(ci_hi_d * 100 + 1, i),
                fontsize=9, va='center')

ax.axvline(baseline_rate * 100, color='#3498db', linestyle='--', linewidth=1.5)
ax.set_yticks(list(y))
ax.set_yticklabels([r['drug'] for _, r in dopa_data.iterrows()], fontsize=10)
ax.set_xlabel('Positive Outcome Rate (%)', fontsize=11)
ax.set_title('Dopaminergic Treatment Forest Plot', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=10,
          handles=[plt.Line2D([0], [0], color='#2ecc71', marker='o', linestyle='', label='Above baseline'),
                   plt.Line2D([0], [0], color='#e74c3c', marker='o', linestyle='', label='Below baseline'),
                   plt.Line2D([0], [0], color='#3498db', linestyle='--', label=f'Baseline ({baseline_rate:.0%})')])
ax.set_xlim(-5, 105)
plt.tight_layout()
plt.show()


**What this means:** Within the dopaminergic class, there is substantial variation. Bupropion and cabergoline show the strongest signals, while generic "d2 agonist" and "dopaminergic drugs" mentions (which may overlap with specific drugs) perform worse. The confidence intervals are wide for all drugs except bupropion, making it difficult to rank them reliably. The key takeaway: dopaminergic augmentation as a strategy shows promise, but drug choice within the class matters.

## Emerging Signals: Diet, Lifestyle, and Psychedelics

Two non-pharmaceutical categories show unexpectedly high positive rates despite small samples. The ketogenic diet (5 users, 100% positive) and psychedelic microdosing (5 users, 88% positive) stand out, but their small n demands extreme caution.

In [ ]:

emerging_drugs = ['ketogenic diet', 'microdosing', 'shrooms', 'lsd', 'weed',
                  'pelvic floor physical therapy', 'tre', 'low dose naltrexone']
emerging = drug_summary[drug_summary['drug'].isin(emerging_drugs)]
emerging = emerging[emerging['n_users'] >= 3].sort_values('pos_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
x_pos = np.arange(len(emerging))
width = 0.25

pos_vals = emerging['pos_rate'].values * 100
mix_vals = (1 - emerging['pos_rate'].values - emerging['neg_rate'].values) * 100
neg_vals = emerging['neg_rate'].values * 100

ax.bar(x_pos - width, pos_vals, width, color='#2ecc71', label='Positive', zorder=2)
ax.bar(x_pos, mix_vals, width, color='#95a5a6', label='Mixed/Neutral', zorder=2)
ax.bar(x_pos + width, neg_vals, width, color='#e74c3c', label='Negative', zorder=2)

for i, (_, row) in enumerate(emerging.iterrows()):
    ci_lo_e, ci_hi_e = row['ci_lo'], row['ci_hi']
    ax.errorbar(i - width, row['pos_rate'] * 100,
                yerr=[[max(0, (row['pos_rate'] - ci_lo_e) * 100)], [(ci_hi_e - row['pos_rate']) * 100]],
                color='black', capsize=3, capthick=1, linewidth=1, zorder=3)

ax.axhline(baseline_rate * 100, color='#3498db', linestyle='--', linewidth=1.5,
           label=f'Baseline ({baseline_rate:.0%})')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{r['drug']}\n(n={r['n_users']})" for _, r in emerging.iterrows()], fontsize=9)
ax.set_ylabel('Rate (%)', fontsize=11)
ax.set_title('Emerging / Non-Pharmaceutical Treatments', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_ylim(0, 115)
plt.tight_layout()
plt.show()

for _, row in emerging.iterrows():
    bt = binomtest(int(row['pos_users']), int(row['n_users']), baseline_rate, alternative='greater')
    if bt.pvalue < 0.10:
        nnt_val = nnt(row['pos_rate'], baseline_rate)
        sig_label = 'Significant at alpha=0.05.' if bt.pvalue < 0.05 else 'Marginally significant at alpha=0.10.'
        nnt_label = nnt_val if nnt_val else "n/a"
        display(HTML(f"<p><b>{row['drug']}:</b> {row['pos_rate']:.0%} positive (n={int(row['n_users'])}), p={bt.pvalue:.4f} vs baseline. {sig_label} NNT={nnt_label}</p>"))


**What this means:** The ketogenic diet's perfect positive rate is striking but based on only 5 users -- the Wilson 95% CI extends down to roughly 57%, meaning the true rate could be much lower. Microdosing psychedelics shows a similar pattern. These are not evidence of efficacy; they are signals worth investigating in larger samples. Low dose naltrexone (LDN), despite only 3 reports, is notable because all are positive and it has an emerging evidence base in autoimmune conditions.

## Counterintuitive Findings Worth Investigating

In [ ]:

cypro = recovery[recovery['drug'] == 'cyproheptadine']
cypro_pos = (cypro['outcome'] == 'positive').sum()
cypro_neg = (cypro['outcome'] == 'negative').sum()
cypro_n = len(cypro)

probi = recovery[recovery['drug'] == 'probiotics']
probi_pos = (probi['outcome'] == 'positive').sum()
probi_total = len(probi)

shroom_full = recovery[recovery['drug'] == 'shrooms']
shroom_micro = recovery[recovery['drug'] == 'microdosing']

micro_pos = (shroom_micro['outcome'] == 'positive').sum()
micro_total = len(shroom_micro)
full_pos = (shroom_full['outcome'] == 'positive').sum()
full_total = max(1, len(shroom_full))

display(HTML(
    '<div style="background: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #f39c12; margin: 10px 0;">'
    '<h3 style="margin-top:0;">1. Cyproheptadine: The Polarizing Serotonin Antagonist</h3>'
    f'<p>Cyproheptadine -- a 5-HT2A/2C antagonist and H1 antihistamine -- splits the community down the middle: {cypro_pos}/{cypro_n} users report positive outcomes, {cypro_neg}/{cypro_n} report negative. For a drug whose mechanism (blocking serotonin receptors) seems tailor-made for PSSD, the 50/50 split is surprising. This may indicate genuine subtype-dependent response: users with persistent serotonin receptor upregulation might benefit, while others with a different underlying mechanism do not. A clinician might expect a serotonin antagonist to be either broadly helpful or broadly unhelpful for PSSD -- not polarizing.</p>'
    '</div>'
    '<div style="background: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #f39c12; margin: 10px 0;">'
    '<h3 style="margin-top:0;">2. Full-Dose Psychedelics Underperform Microdosing</h3>'
    f'<p>Microdosing shows {micro_pos}/{micro_total} positive reports ({micro_pos/micro_total:.0%}), while full-dose psilocybin ("shrooms") shows only {full_pos}/{full_total} ({full_pos/full_total:.0%}). The community enthusiasm for psychedelics as a PSSD treatment rests on the theory that they promote neuroplasticity. If that theory is correct, one would expect higher doses to produce stronger effects -- yet the opposite pattern appears here. Possible explanations: microdosing users may be more systematic (regular protocol over weeks), while full-dose users may try a single session; or the populations differ in other ways. The sample sizes (n={micro_total} and n={len(shroom_full)}) are too small for a formal comparison.</p>'
    '</div>'
    '<div style="background: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #f39c12; margin: 10px 0;">'
    '<h3 style="margin-top:0;">3. Probiotics: Popular but Disappointing</h3>'
    f'<p>Probiotics have {probi_total} user reports -- the gut-brain axis hypothesis is popular in the PSSD community. Yet the positive rate is just {probi_pos}/{probi_total} ({probi_pos/probi_total:.0%}), well below the community baseline. For a treatment that the community frequently recommends and discusses, this is a notable underperformance. The gut-brain narrative may be driving adoption despite mediocre outcomes in practice.</p>'
    '</div>'
))


## What Patients Are Saying *(experimental)*

The following quotes are drawn from posts by users who reported on top recovery treatments. Each quote contains a specific treatment outcome and is dated for context.

In [ ]:

display(HTML(
    '<div style="background: #e8f5e9; padding: 12px; border-radius: 8px; margin: 8px 0; border-left: 3px solid #2ecc71;">'
    '<b>Bupropion -- positive response (2026-03-16):</b><br>'
    '<em>"I have started 75mg [bupropion] from last four days. It works for me. Sexual activity has started which was down due to SSRIs."</em>'
    '</div>'
    '<div style="background: #fce4ec; padding: 12px; border-radius: 8px; margin: 8px 0; border-left: 3px solid #e74c3c;">'
    '<b>Bupropion -- dose-dependent worsening (2026-03-18):</b><br>'
    '<em>"I saw improvement with 150 mg, but 300 mg worsened my anhedonia and led to ED, which I still suffer from."</em>'
    '</div>'
    '<div style="background: #e8f5e9; padding: 12px; border-radius: 8px; margin: 8px 0; border-left: 3px solid #2ecc71;">'
    '<b>Antihistamine/Mast cell protocol -- full recovery claim (2026-03-17):</b><br>'
    '<em>"I consider myself healed, now. [...] I did Immunoadsorptions, one PLEX, took LDN and Antihistamines (Loratadin, Ketotifen, Famotidin) and took ONLY one pill of liposomal quercetin."</em>'
    '</div>'
    '<div style="background: #e8f5e9; padding: 12px; border-radius: 8px; margin: 8px 0; border-left: 3px solid #2ecc71;">'
    '<b>Ketogenic diet -- partial improvement (2026-03-13):</b><br>'
    '<em>"About 3 months into my crash I started keto and I began feeling much better about 2 months into keto, still very symptomatic but I was able to focus on my life."</em>'
    '</div>'
    '<div style="background: #fff8e1; padding: 12px; border-radius: 8px; margin: 8px 0; border-left: 3px solid #f39c12;">'
    '<b>Cyproheptadine -- mixed/subtype-dependent (2026-03-29):</b><br>'
    '<em>"I improved on it greatly but had to stop because of OCD. [...] I believe some PSSD people have persistent 5HT2 overactivation, so if you are in that subtype it should help."</em>'
    '</div>'
))


## Tiered Recommendations

Treatments are classified by evidence strength: **Strong** (n >= 30, p < 0.05), **Moderate** (n >= 15 or p < 0.10), **Preliminary** (n < 15). The positive rate, confidence interval, and mechanism group are shown for context.

In [ ]:

tier_data = drug_summary[drug_summary['n_users'] >= 3].copy()
tier_data = tier_data[tier_data['pos_rate'] > 0]

def assign_tier(row):
    n = row['n_users']
    bt = binomtest(int(row['pos_users']), int(n), baseline_rate, alternative='greater')
    p = bt.pvalue
    if n >= 30 and p < 0.05:
        return 'Strong'
    elif n >= 15 or p < 0.10:
        return 'Moderate'
    else:
        return 'Preliminary'

tier_data['tier'] = tier_data.apply(assign_tier, axis=1)
tier_data['p_val'] = tier_data.apply(
    lambda r: binomtest(int(r['pos_users']), int(r['n_users']), baseline_rate, alternative='greater').pvalue, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#3498db'}

for idx, tier in enumerate(['Strong', 'Moderate', 'Preliminary']):
    ax = axes[idx]
    t_data = tier_data[tier_data['tier'] == tier].sort_values('pos_rate', ascending=True)

    if len(t_data) == 0:
        ax.text(0.5, 0.5, f'No {tier}\nrecommendations', ha='center', va='center',
                fontsize=12, transform=ax.transAxes)
        ax.set_title(f'{tier} Evidence', fontsize=12, fontweight='bold', color=tier_colors[tier])
        ax.set_xlim(0, 100)
        continue

    y_t = range(len(t_data))
    for i, (_, row) in enumerate(t_data.iterrows()):
        ci_lo_t, ci_hi_t = row['ci_lo'], row['ci_hi']
        ax.plot([ci_lo_t * 100, ci_hi_t * 100], [i, i], color='#555', linewidth=1.5, zorder=1)
        ax.scatter(row['pos_rate'] * 100, i, c=tier_colors[tier], s=80, zorder=2, edgecolors='white')

    ax.axvline(baseline_rate * 100, color='#3498db', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_yticks(list(y_t))
    ax.set_yticklabels([f"{r['drug']} (n={r['n_users']})" for _, r in t_data.iterrows()], fontsize=9)
    ax.set_title(f'{tier} Evidence', fontsize=12, fontweight='bold', color=tier_colors[tier])
    ax.set_xlabel('Positive Rate (%)', fontsize=10)
    ax.set_xlim(-5, 105)

plt.suptitle('Treatment Recommendations by Evidence Tier', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

for tier in ['Strong', 'Moderate', 'Preliminary']:
    t_subset = tier_data[tier_data['tier'] == tier].sort_values('pos_rate', ascending=False)
    if len(t_subset) > 0:
        display(HTML(f"<h4>{tier} Evidence ({len(t_subset)} treatments)</h4>"))
        disp_df = t_subset[['drug', 'mechanism', 'n_users', 'pos_rate', 'ci_lo', 'ci_hi', 'p_val']].copy()
        disp_df.columns = ['Treatment', 'Mechanism', 'Users', 'Pos Rate', 'CI Low', 'CI High', 'p-value']
        disp_df['Pos Rate'] = disp_df['Pos Rate'].apply(lambda x: f"{x:.0%}")
        disp_df['CI'] = disp_df.apply(lambda r: f"{r['CI Low']:.0%} to {r['CI High']:.0%}", axis=1)
        disp_df = disp_df[['Treatment', 'Mechanism', 'Users', 'Pos Rate', 'CI', 'p-value']]
        styled = disp_df.style.format({'p-value': '{:.4f}'}).set_properties(**{'font-size': '11px'})
        display(styled)


## Conclusion

The r/PSSD community's one-month snapshot reveals a recovery landscape dominated by trial and error, with a community baseline positive rate that falls well below 50%. Most treatments people try do not help, which aligns with the clinical reality that PSSD remains poorly understood and has no established treatment protocol.

That said, several signal clusters emerge. **Antihistamines and mast cell stabilizers** represent the most coherent recovery strategy in this data. The convergence of loratadine, ketotifen, cyproheptadine, quercetin, and LDN around the mast cell hypothesis -- and the existence of at least one detailed recovery account using a combined mast cell protocol -- gives this class face validity beyond what the numbers alone provide. A patient asking "what should I try?" would find the antihistamine class the most promising starting point based on this data.

**Bupropion** is the single most evidence-supported individual treatment, with the largest sample size and a positive rate that is meaningfully above baseline. Its dopaminergic mechanism provides a clear pharmacological rationale for PSSD (replacing dopaminergic tone suppressed by serotonergic drugs). However, dose matters: the community reports suggest low doses (75-150mg) are more helpful than high doses (300mg), which may worsen anhedonia. A patient considering bupropion should start low and monitor closely.

**The ketogenic diet and psychedelic microdosing** show the most striking raw numbers but the weakest evidence base. Their perfect or near-perfect positive rates in samples of 5 users are likely inflated by small-n noise and reporting bias (people who did not benefit may not post about it). These deserve further investigation -- particularly the ketogenic diet, which has no serious safety concerns -- but should not be presented as established treatments.

The most important finding may be the counterintuitive one: **cyproheptadine's polarizing response pattern** suggests that PSSD is not a single condition but a syndrome with subtypes. If serotonin receptor antagonism helps exactly half of patients and fails the other half, the field may benefit more from subtyping patients than from testing more drugs. Future research should investigate what distinguishes cyproheptadine responders from non-responders.

## Research Limitations

1. **Selection bias:** Reddit users skew younger, male, English-speaking, and tech-savvy. The r/PSSD community is a self-selected group of people sufficiently affected and motivated to seek online support. Results do not generalize to all PSSD patients.

2. **Reporting bias:** Users are more likely to post about dramatic experiences (recovery stories, severe side effects) than about treatments that had no effect. This inflates both the positive and negative tails while underrepresenting null outcomes.

3. **Survivorship bias:** Users who recover may leave the community, reducing their representation in ongoing discussions. Conversely, users who worsen may become more active posters. Long-term recovery rates may be underestimated.

4. **Recall bias:** Treatment reports are retrospective self-assessments, not contemporaneous clinical observations. Users may misattribute improvement to the wrong treatment, especially when trying multiple interventions simultaneously.

5. **Confounding:** Many users try multiple treatments concurrently or sequentially. Positive outcomes attributed to one drug may reflect the combined effect of several, natural recovery over time, or placebo response. We cannot isolate individual treatment effects.

6. **No control group:** There is no untreated comparison group. The baseline positive rate includes natural recovery, placebo effects, and regression to the mean. We cannot determine what proportion of "positive" outcomes would have occurred without treatment.

7. **Sentiment vs. efficacy:** Our outcome measure is sentiment extracted from text, not clinical endpoints. A "positive" sentiment report may reflect partial improvement, hopeful expectation, or relief at trying something new -- not necessarily objective sexual function recovery.

8. **Temporal snapshot:** One month of data (March-April 2026) may not capture seasonal patterns, evolving community opinions, or treatments that require longer observation periods. Conclusions are provisional and may not replicate in a different time window.

In [ ]:
display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; padding: 20px; margin-top: 20px; background: #f8d7da; border-radius: 8px; text-align: center;">These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.</div>'))

In [ ]:
conn.close()